# 🏠 Swiss Mortgage Simulator — 2 Tranches
Assumes **20% down payment** and **5% notary fees**.  
Monthly interest charged on outstanding principal (*Festhypothek*).  
Principal amortised at **1% p.a.** of original mortgage (pro-rata per tranche).  
**Bonus Vert**: waives Tranche 1 interest for the first 12 months.

In [24]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntText, FloatText, IntSlider, Checkbox, HBox, VBox, HTML
import ipywidgets as widgets
import warnings
warnings.filterwarnings("ignore")

import matplotlib as mpl
mpl.rcParams.update({
    "axes.titlesize": 22,
    "axes.labelsize": 22,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "legend.fontsize": 22,
})

# ── colour palette ───────────────────────────────────────────────
C1      = '#2563EB'   # tranche 1  – blue
C2      = '#16A34A'   # tranche 2  – green
C_AMORT = '#F59E0B'   # amortisation – amber
C_TOTAL = '#64748B'   # total – slate
C_SAVE  = '#7C3AED'   # savings / cumulative – violet
C_DIFFP = '#ef4444'   # positive diff (two‑tranche more expensive) – red
C_DIFFN = '#22c55e'   # negative diff (two‑tranche cheaper) – green
C_BONUS = '#10B981'   # bonus highlight
BG      = '#F8FAFC'
GRID    = '#E2E8F0'

# ── widgets ──────────────────────────────────────────────────────
w_price  = IntText(value=995_000, description='Property (CHF)',
                   style={'description_width':'130px'}, layout=widgets.Layout(width='280px'))

w_rate1  = FloatText(value=1.8,  description='Rate T1 (%)',
                     style={'description_width':'130px'}, layout=widgets.Layout(width='240px'), step=0.05)
w_rate2  = FloatText(value=2.2,  description='Rate T2 (%)',
                     style={'description_width':'130px'}, layout=widgets.Layout(width='240px'), step=0.05)
w_rate_cmp = FloatText(value=2.0, description='Rate compare (%)',
                       style={'description_width':'130px'}, layout=widgets.Layout(width='240px'), step=0.05)

w_years1 = IntSlider(value=5, min=2, max=10, step=1, description='Years T1',
                     style={'description_width':'100px'}, layout=widgets.Layout(width='350px'))
w_years2 = IntSlider(value=5, min=2, max=10, step=1, description='Years T2',
                     style={'description_width':'100px'}, layout=widgets.Layout(width='350px'))
w_split  = FloatText(value=100.0, description='T1 share (%)',
                     style={'description_width':'130px'}, layout=widgets.Layout(width='240px'), step=5.0)

w_bonus  = Checkbox(value=False, description='🌿 Bonus Vert (waive T1 interest – Year 1)',
                    style={'description_width':'initial'}, layout=widgets.Layout(width='420px'))

header = HTML('<h3 style="font-family:Georgia,serif;margin-bottom:4px">🏔 Swiss Mortgage Simulator</h3>')
note   = HTML(
    '<p style="color:#64748b;font-size:0.85em;font-family:Georgia,serif">'
    'Down payment 20% · Notary fees 5% · Festhypothek · '
    'Amortisation 1.24% p.a. of original mortgage (pro‑rata per tranche) · '
    'Bonus Vert toggled via checkbox · '
    'Rate compare: single‑tranche @ specified rate for 10y</p>'
)

controls = VBox([
    header, note,
    HBox([w_price]),
    HBox([w_rate1, w_rate2, w_rate_cmp, w_split]),
    HBox([w_years1]),
    HBox([w_years2]),
    HBox([w_bonus]),
])

out = widgets.Output()

# ── core simulation ───────────────────────────────────────────────
def simulate(price, rate1, rate2, years1, years2, split_pct):
    DOWN       = 0.20
    NOTARY     = 0.05
    AMORT_RATE = 0.0124  # 1.24% p.a. of original mortgage

    total_cost   = price
    down_payment = price * DOWN
    mortgage     = total_cost - down_payment

    split = np.clip(split_pct, 1, 99) / 100
    p1_0  = mortgage * split
    p2_0  = mortgage * (1 - split)

    monthly_amort1 = mortgage * AMORT_RATE * split       / 12
    monthly_amort2 = mortgage * AMORT_RATE * (1 - split) / 12

    months = 120
    m1 = years1 * 12
    m2 = years2 * 12
    r1 = rate1 / 100 / 12
    r2 = rate2 / 100 / 12

    def run(apply_bonus):
        p1 = np.zeros(months + 1);  p1[0] = p1_0
        p2 = np.zeros(months + 1);  p2[0] = p2_0
        ic1 = np.zeros(months)
        ic2 = np.zeros(months)
        am  = np.zeros(months)
        for i in range(months):
            v1 = max(p1[i], 0.0)
            v2 = max(p2[i], 0.0)
            eff_r1 = r1 if (i // m1) == 0 else r2  # T1 rolls to T2 rate after its term
            eff_r2 = r2

            # Bonus Vert: zero T1 interest for first 12 months
            ic1[i] = 0.0 if (apply_bonus and i < 12) else v1 * eff_r1
            ic2[i] = v2 * eff_r2

            a1 = min(monthly_amort1, v1)
            a2 = min(monthly_amort2, v2)
            am[i] = a1 + a2

            p1[i+1] = v1 - a1
            p2[i+1] = v2 - a2

        return ic1, ic2, am, p1[:months], p2[:months]

    # Precompute both scenarios
    ic1_bonus, ic2_bonus, am_bonus, pr1_bonus, pr2_bonus = run(apply_bonus=True)
    ic1_base,  ic2_base,  am_base,  pr1_base,  pr2_base  = run(apply_bonus=False)

    total_bonus = ic1_bonus + ic2_bonus + am_bonus
    total_base  = ic1_base  + ic2_base  + am_base

    savings  = total_base - total_bonus
    cum_save = np.cumsum(savings)

    return dict(
        mortgage       = mortgage,
        down           = down_payment,
        notary         = price * NOTARY,
        p1_0=p1_0, p2_0=p2_0,
        monthly_amort  = mortgage * AMORT_RATE / 12,
        # both scenarios
        ic1_bonus=ic1_bonus, ic2_bonus=ic2_bonus, am_bonus=am_bonus, total_bonus=total_bonus,
        ic1_base=ic1_base,   ic2_base=ic2_base,   am_base=am_base,   total_base=total_base,
        principal1_bonus=pr1_bonus, principal2_bonus=pr2_bonus,
        principal1_base=pr1_base,   principal2_base=pr2_base,
        # savings vs baseline
        savings=savings, cum_save=cum_save, total_bonus_saving=cum_save[-1],
        months=months, m1=m1, m2=m2,
    )

def single_tranche_compare_series(mortgage, rate_compare_pct, months, amort_rate_annual):
    """Monthly total (interest + amort) for a single-tranche mortgage at rate_compare over 'months'."""
    r_cmp = (rate_compare_pct / 100.0) / 12.0
    monthly_amort = mortgage * amort_rate_annual / 12.0
    p = np.zeros(months + 1); p[0] = mortgage
    ic = np.zeros(months)
    tot = np.zeros(months)
    for i in range(months):
        bal = max(p[i], 0.0)
        ic[i] = bal * r_cmp
        a = min(monthly_amort, bal)
        tot[i] = ic[i] + a
        p[i+1] = bal - a
    return tot  # length 'months'

# ── plot ──────────────────────────────────────────────────────────
def update(*args):
    out.clear_output(wait=True)
    price     = w_price.value
    rate1     = w_rate1.value
    rate2     = w_rate2.value
    rate_cmp  = w_rate_cmp.value
    years1    = w_years1.value
    years2    = w_years2.value
    split_pct = w_split.value
    bonus_on  = w_bonus.value

    if price <= 0 or rate1 <= 0 or rate2 <= 0 or rate_cmp <= 0:
        return

    d          = simulate(price, rate1, rate2, years1, years2, split_pct)
    time_years = np.arange(d['months']) / 12
    xticks     = np.arange(0, 11, 1)

    # Choose the "active" scenario for plots 1,2 and rate-compare
    if bonus_on:
        ic1 = d['ic1_bonus']; ic2 = d['ic2_bonus']; am = d['am_bonus']; total = d['total_bonus']
        pr1 = d['principal1_bonus']; pr2 = d['principal2_bonus']
    else:
        ic1 = d['ic1_base'];  ic2 = d['ic2_base'];  am = d['am_base'];  total = d['total_base']
        pr1 = d['principal1_base']; pr2 = d['principal2_base']

    # Single-tranche comparison series (compare vs current active scenario)
    cmp_total = single_tranche_compare_series(
        mortgage=d['mortgage'],
        rate_compare_pct=rate_cmp,
        months=d['months'],
        amort_rate_annual=0.0124
    )
    diff_monthly = total - cmp_total        # >0 means current scenario costs more
    cum_diff     = np.cumsum(diff_monthly)

    with out:
        fig, axes = plt.subplots(4, 1, figsize=(20, 4*9.0), facecolor=BG)
        fig.subplots_adjust(wspace=0.38)

        bonus_tag = "ON" if bonus_on else "OFF"
        fig.suptitle(
            f"  Purchase: CHF {price:,.0f}  │  Notary: CHF {d['notary']:,.0f}  │  "
            f"Down: CHF {d['down']:,.0f}  │  Mortgage: CHF {d['mortgage']:,.0f}  │  "
            f"T1: CHF {d['p1_0']:,.0f} @ {rate1:.2f}%  │  "
            f"T2: CHF {d['p2_0']:,.0f} @ {rate2:.2f}%  │  "
            f"Bonus Vert: {bonus_tag}  │  "
            f"Compare: single‑tranche @ {rate_cmp:.2f}% for 10y",
            fontsize=22.0, color='#475569', fontfamily='monospace', y=1.02
        )

        def add_renewal_markers(ax, ymax):
            for k in range(1, 11):
                t = k * years1
                if 0 < t < 10:
                    ax.axvline(t, color=C1, linewidth=1, linestyle='--', alpha=0.5)
                    ax.text(t + 0.06, ymax * 0.97, 'T1↺', fontsize=22, color=C1, va='top')
            for k in range(1, 11):
                t = k * years2
                if 0 < t < 10:
                    ax.axvline(t, color=C2, linewidth=1, linestyle=':', alpha=0.5)
                    ax.text(t + 0.06, ymax * 0.85, 'T2↺', fontsize=22, color=C2, va='top')

        # ── Plot 1: stacked monthly cost (active scenario) ─
        ax1 = axes[0]
        ax1.set_facecolor(BG)
        ax1.stackplot(
            time_years,
            ic1, ic2, am,
            labels=[f"Interest T1{' (Bonus Year 1)' if bonus_on else ''}", 'Interest T2', 'Amortisation (1.24% p.a.)'],
            colors=[C1, C2, C_AMORT], alpha=0.75
        )
        ax1.plot(time_years, total, color=C_TOTAL, linewidth=1.8, linestyle='--', label='Total')
        if bonus_on:
            ax1.axvspan(0, 1, color=C_BONUS, alpha=0.08, zorder=0)
            ax1.text(0.03, max(1.0, total.max()) * 1.05, '🌿 Bonus Vert',
                     fontsize=22, color=C_BONUS, va='bottom')
        ymax1 = total.max() * 1.15
        add_renewal_markers(ax1, ymax1)
        ax1.set_xlim(0, 10); ax1.set_ylim(0, ymax1)
        ax1.set_xticks(xticks)
        ax1.set_xlabel('Year', fontsize=22)
        ax1.set_ylabel('CHF / month', fontsize=22)
        ax1.set_title(f"Monthly Cost Breakdown — Bonus {bonus_tag}", fontsize=22, fontweight='bold', pad=10)
        ax1.legend(fontsize=22, framealpha=0.6, loc='upper right')
        ax1.grid(True, color=GRID, linewidth=0.8, zorder=0)
        ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

        # ── Plot 2: outstanding principal (active scenario) ─
        ax2 = axes[1]
        ax2.set_facecolor(BG)
        total_p = (pr1 + pr2) / 1000
        ax2.fill_between(time_years, pr1 / 1000, alpha=0.15, color=C1)
        ax2.fill_between(time_years, pr2 / 1000, alpha=0.15, color=C2)
        ax2.plot(time_years, pr1 / 1000, color=C1, linewidth=2.5,
                 label=f'Tranche 1  ({rate1:.2f}%,  {years1}yr)')
        ax2.plot(time_years, pr2 / 1000, color=C2, linewidth=2.5,
                 label=f'Tranche 2  ({rate2:.2f}%,  {years2}yr)')
        ax2.plot(time_years, total_p, color=C_TOTAL, linewidth=1.8, linestyle='--', label='Total')
        ax2.set_xlim(0, 10); ax2.set_xticks(xticks)
        ax2.set_xlabel('Year', fontsize=22)
        ax2.set_ylabel('Outstanding Principal (CHF k)', fontsize=22)
        ax2.set_title('Outstanding Principal per Tranche', fontsize=22, fontweight='bold', pad=10)
        ax2.legend(fontsize=22, framealpha=0.5)
        ax2.grid(True, color=GRID, linewidth=0.8)
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}k'))

        # ── Plot 3: Bonus Vert — savings vs baseline (no bonus) ─
        ax3 = axes[2]
        ax3.set_facecolor(BG)

        # Savings series always shown; will be ~0 when bonus is OFF
        bar_colors = [C_BONUS if s > 0 else '#E2E8F0' for s in d['savings']]
        ax3.bar(time_years, d['savings'], width=1/12, color=bar_colors, alpha=0.6, label='Monthly saving')

        ax3b = ax3.twinx()
        ax3b.plot(time_years, d['cum_save'], color=C_SAVE, linewidth=2.5, label='Cumulative saving')
        ax3b.fill_between(time_years, d['cum_save'], alpha=0.10, color=C_SAVE)
        ax3b.set_ylabel('Cumulative Saving (CHF)', fontsize=22, color=C_SAVE)
        ax3b.tick_params(axis='y', colors=C_SAVE)
        ax3b.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        ymax_cum = d['cum_save'].max()
        ax3b.set_ylim(0, ymax_cum * 1.35 if ymax_cum > 0 else 1)

        total_saved = d['total_bonus_saving']
        ax3b.annotate(
            f"Total saved (10y):\nCHF {total_saved:,.0f}",
            xy=(1.0, total_saved),
            xytext=(3.5, (total_saved * 1.15) if total_saved > 0 else 1.0),
            fontsize=22, color=C_SAVE, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C_SAVE, lw=1.5)
        )

        if bonus_on:
            ax3.axvspan(0, 1, color=C_BONUS, alpha=0.07, zorder=0)
            ax3.axvline(1, color=C_BONUS, linewidth=1.2, linestyle='--', alpha=0.7)

        ax3.set_xlim(0, 10); ax3.set_xticks(xticks)
        ax3.set_xlabel('Year', fontsize=22)
        ax3.set_ylabel('Monthly Saving (CHF)', fontsize=22)
        ax3.set_title('🌿 Bonus Vert — Savings vs Standard Mortgage', fontsize=22, fontweight='bold', pad=10)
        ax3.grid(True, color=GRID, linewidth=0.8, zorder=0)
        ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        h1, l1 = ax3.get_legend_handles_labels()
        h2, l2 = ax3b.get_legend_handles_labels()
        ax3.legend(h1 + h2, l1 + l2, fontsize=16, framealpha=0.6, loc='center right')
        ax3b.spines['right'].set_edgecolor(C_SAVE)

        # ── Plot 4: Rate compare — active scenario vs single‑tranche ─
        ax4 = axes[3]
        ax4.set_facecolor(BG)

        bar_colors4 = [C_DIFFP if v > 0 else C_DIFFN for v in diff_monthly]
        ax4.bar(time_years, diff_monthly, width=1/12, color=bar_colors4, alpha=0.6, label='Monthly difference')

        ax4b = ax4.twinx()
        ax4b.plot(time_years, cum_diff, color=C_SAVE, linewidth=2.5, label='Cumulative difference')
        ax4b.fill_between(time_years, cum_diff, alpha=0.10, color=C_SAVE)
        ax4b.set_ylabel('Cumulative Difference (CHF)', fontsize=22, color=C_SAVE)
        ax4b.tick_params(axis='y', colors=C_SAVE)
        ax4b.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

        y_abs = max(1.0, np.max(np.abs(diff_monthly)))
        ax4.set_ylim(-y_abs * 1.3, y_abs * 1.3)
        ax4b.set_ylim(min(0, cum_diff.min()) * 1.15, max(0, cum_diff.max()) * 1.35)

        final_cum = cum_diff[-1]
        ax4b.annotate(
            f"Total after 10y:\nCHF {final_cum:,.0f}",
            xy=(time_years[-1], final_cum),
            xytext=(max(1.0, time_years[-1] - 3.0), final_cum + (0.15 * (ax4b.get_ylim()[1]-ax4b.get_ylim()[0]))),
            fontsize=22, color=C_SAVE, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C_SAVE, lw=1.5)
        )

        ax4.set_xlim(0, 10); ax4.set_xticks(xticks)
        ax4.set_xlabel('Year', fontsize=22)
        ax4.set_ylabel('Monthly Difference (CHF)', fontsize=22)
        ax4.set_title(f'Rate Compare — Active vs Single‑tranche @ {rate_cmp:.2f}%',
                      fontsize=22, fontweight='bold', pad=10)
        ax4.grid(True, color=GRID, linewidth=0.8, zorder=0)
        ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        h1, l1 = ax4.get_legend_handles_labels()
        h2, l2 = ax4b.get_legend_handles_labels()
        ax4.legend(h1 + h2, l1 + l2, fontsize=16, framealpha=0.6, loc='center right')
        ax4b.spines['right'].set_edgecolor(C_SAVE)

        # Aesthetics: gridline color on spines
        for ax in [axes[0], axes[1], axes[2], axes[3]]:
            for spine in ax.spines.values():
                spine.set_edgecolor(GRID)

        plt.tight_layout()
        plt.show()

# ── wire up observers ─────────────────────────────────────────────
for w in [w_price, w_rate1, w_rate2, w_rate_cmp, w_years1, w_years2, w_split, w_bonus]:
    w.observe(update, names='value')

display(controls, out)
update()


Output()